In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Find the project root, regardless of the notebook's working directory.
candidates = [Path.cwd(), *Path.cwd().parents]

PROJECT_ROOT = next(
    (
        path for path in candidates
        if (
            path / "data" / "processed" / "credit_risk_cohort.csv"
        ).exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find data/processed/credit_risk_cohort.csv. "
        "Check that the preparation script ran successfully."
    )

DATA_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "credit_risk_cohort.csv"
)

REPORT_DIR = PROJECT_ROOT / "reports" / "credit_risk"
FIGURE_DIR = REPORT_DIR / "figures"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

credit = pd.read_csv(DATA_FILE, parse_dates=["issue_date"])

print("Dataset shape:", credit.shape)
print("\nFirst five records:")
display(credit.head())

print("\nTarget distribution:")
display(credit["bad_outcome"].value_counts(dropna=False))

print(
    "\nDuplicate record IDs:",
    credit["record_id"].duplicated().sum()
)

print(
    "\nMissing target values:",
    credit["bad_outcome"].isna().sum()
)

In [ ]:
total_loans = len(credit)
bad_loans = int(credit["bad_outcome"].sum())
good_loans = total_loans - bad_loans

bad_rate = credit["bad_outcome"].mean() * 100

print(f"Resolved loans: {total_loans:,}")
print(f"Fully paid loans: {good_loans:,}")
print(f"Charged-off loans: {bad_loans:,}")
print(f"Historical bad-outcome rate: {bad_rate:.2f}%")

In [ ]:
summary = (
    credit.groupby("bad_outcome")
    .agg(
        loan_count=("record_id", "size"),
        original_loan_amount=("loan_amnt", "sum"),
        average_loan_amount=("loan_amnt", "mean"),
        average_income=("annual_inc", "mean"),
    )
    .reset_index()
)

summary["loan_share_pct"] = (
    summary["loan_count"] / total_loans * 100
).round(2)

summary["original_amount_share_pct"] = (
    summary["original_loan_amount"]
    / credit["loan_amnt"].sum()
    * 100
).round(2)

summary["outcome"] = summary["bad_outcome"].map({
    0: "Fully paid",
    1: "Charged off",
})

display(summary)

In [ ]:
import math


def wilson_interval(bad_count, total_count, z=1.96):
    """
    Approximate 95% Wilson confidence interval for a proportion.
    Returns percentages.
    """
    if total_count == 0:
        return np.nan, np.nan

    p = bad_count / total_count
    denominator = 1 + (z**2 / total_count)

    center = (
        p + z**2 / (2 * total_count)
    ) / denominator

    margin = (
        z
        * math.sqrt(
            p * (1 - p) / total_count
            + z**2 / (4 * total_count**2)
        )
        / denominator
    )

    lower = max(0, center - margin) * 100
    upper = min(1, center + margin) * 100

    return lower, upper


def analyze_segment(df, segment_column, minimum_n=100):
    """
    Compare observed outcomes across segments.

    minimum_n prevents very small groups from dominating
    the displayed ranking.
    """
    rows = []

    for segment, group in df.groupby(
        segment_column,
        dropna=False,
        observed=True
    ):
        n = len(group)

        if n < minimum_n:
            continue

        bad = int(group["bad_outcome"].sum())
        rate = bad / n * 100

        low, high = wilson_interval(bad, n)

        rows.append({
            "segment": (
                "Missing"
                if pd.isna(segment)
                else str(segment)
            ),
            "loan_count": n,
            "bad_loans": bad,
            "bad_rate_pct": round(rate, 2),
            "ci_95_low_pct": round(low, 2),
            "ci_95_high_pct": round(high, 2),
            "bad_loan_share_pct": round(
                bad / df["bad_outcome"].sum() * 100,
                2,
            ) if df["bad_outcome"].sum() else np.nan,
            "original_loan_amount": group["loan_amnt"].sum(),
        })

    return (
        pd.DataFrame(rows)
        .sort_values("bad_rate_pct", ascending=False)
        .reset_index(drop=True)
    )

In [ ]:
grade_results=analyze_segment(
    credit,
    "grade",
    minimum_n=100,
)
display(grade_results)

In [ ]:
# Exclude the missing-grade group from the grade ranking.
# We want to compare actual credit grades.

grade_ranked = grade_results[
    grade_results["segment"] != "Missing"
].sort_values(
    "bad_rate_pct",
    ascending=False
)

display(grade_ranked)

# Identify the grade with the highest observed charged-off rate.
top_grade = grade_ranked.iloc[0]

print("\nHighest-risk grade in the resolved-loan sample")
print("----------------------------------------------")
print("Credit grade:", top_grade["segment"])
print("Number of loans:", int(top_grade["loan_count"]))
print("Charged-off loans:", int(top_grade["bad_loans"]))
print("Charged-off rate:", f'{top_grade["bad_rate_pct"]:.2f}%')
print(
    "Share of all charged-off loans:",
    f'{top_grade["bad_loan_share_pct"]:.2f}%'
)
print(
    "95% confidence interval:",
    f'{top_grade["ci_95_low_pct"]:.2f}%–'
    f'{top_grade["ci_95_high_pct"]:.2f}%'
)

In [ ]:
grade_results.to_csv(
    REPORT_DIR / "risk_by_grade.csv",
    index=False,
)

In [ ]:
plot_data = grade_results.sort_values("segment")

plt.figure(figsize=(9, 5))
plt.bar(
    plot_data["segment"],
    plot_data["bad_rate_pct"],
)

plt.axhline(
    bad_rate,
    linestyle="--",
    label=f"Portfolio baseline: {bad_rate:.2f}%",
)

plt.title("Historical Charged-Off Rate by Credit Grade")
plt.xlabel("Credit grade")
plt.ylabel("Charged-off loans (%)")
plt.legend()
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "bad_rate_by_grade.png",
    dpi=160,
    bbox_inches="tight",
)

plt.show()

In [ ]:
for column in [
    "purpose",
    "home_ownership",
    "term",
    "verification_status",
]:
    if column not in credit.columns:
        print(f"\nSkipping unavailable column: {column}")
        continue

    results = analyze_segment(
        credit,
        column,
        minimum_n=100,
    )

    print(f"\n===== BAD-OUTCOME RATE BY {column.upper()} =====")
    display(results)

    results.to_csv(
        REPORT_DIR / f"risk_by_{column}.csv",
        index=False,
    )

In [ ]:
def analyze_numeric_bands(df, column, bins=5):
    temp = df[[column, "bad_outcome"]].copy()

    temp[column] = pd.to_numeric(
        temp[column],
        errors="coerce",
    )

    temp = temp.dropna(subset=[column])

    if temp.empty or temp[column].nunique() < 2:
        print(f"Not enough valid variation in {column}")
        return pd.DataFrame()

    temp["band"] = pd.qcut(
        temp[column],
        q=bins,
        duplicates="drop",
    )

    result = (
        temp.groupby("band", observed=True)
        .agg(
            loan_count=("bad_outcome", "size"),
            bad_loans=("bad_outcome", "sum"),
            average_feature_value=(column, "mean"),
        )
        .reset_index()
    )

    result["bad_rate_pct"] = (
        result["bad_loans"]
        / result["loan_count"]
        * 100
    ).round(2)

    result["band"] = result["band"].astype(str)

    return result

In [ ]:
numeric_features = [
    "annual_inc",
    "dti",
    "revol_util",
    "loan_amnt",
    "delinq_2yrs",
]

for column in numeric_features:
    if column not in credit.columns:
        print(f"Skipping unavailable feature: {column}")
        continue

    result = analyze_numeric_bands(credit, column)

    print(f"\n===== RISK BY {column.upper()} =====")
    display(result)

    result.to_csv(
        REPORT_DIR / f"risk_by_{column}_band.csv",
        index=False,
    )

In [ ]:
from pathlib import Path
import pandas as pd

report_dir = PROJECT_ROOT / "reports" / "credit_risk"

files = [
    "risk_by_dti_band.csv",
    "risk_by_annual_inc_band.csv",
    "risk_by_revol_util_band.csv",
]

for filename in files:
    filepath = report_dir / filename

    print(f"\n{'=' * 12} {filename} {'=' * 12}")

    if filepath.exists():
        display(pd.read_csv(filepath))
    else:
        print("File not found. Rerun the numerical-band analysis cell.")

In [ ]:
for filename in [
    "risk_by_dti_band.csv",
    "risk_by_annual_inc_band.csv",
    "risk_by_revol_util_band.csv",
]:
    df = pd.read_csv(REPORT_DIR / filename)

    df["recalculated_rate_pct"] = (
        df["bad_loans"] / df["loan_count"] * 100
    ).round(2)

    print(f"\n{filename}")
    display(df[[
        "band",
        "loan_count",
        "bad_loans",
        "bad_rate_pct",
        "recalculated_rate_pct",
    ]])